# Module 1: Validation & Go/No-Go Checklist

## Overview

This notebook validates that your LangSmith deployment is healthy and ready for use. This checklist becomes your **baseline reference** for future troubleshooting.

### What We'll Validate

1. ✅ Pod readiness (all pods running)
2. ✅ PVC binding (storage provisioned)
3. ✅ Ingress provisioning (ALB created)
4. ✅ Endpoint reachability (services accessible)
5. ✅ Basic UI availability (web interface works)

### Why This Matters

Most issues are caught here, before real users onboard. This validation ensures you're on a **supported path**.

**Estimated time:** 20-30 minutes


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
# Find the notebooks directory by looking for the shared folder
possible_paths = [
    Path.cwd().parent,  # If cwd is module-1, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## Setting Up Cluster Access

Ensure kubectl is configured for the EKS cluster.


In [ ]:
import os
from shared._validation import require_env, ok
from shared._cloud_helpers import (
    get_cloud_provider,
    get_region,
    configure_kubectl,
)

provider = get_cloud_provider()

# Cloud-specific region variable
if provider == "aws":
    region_var = "AWS_REGION"
elif provider == "azure":
    region_var = "AZURE_LOCATION"
else:
    region_var = "AWS_REGION"  # Default for backward compatibility

# Get configuration
config = require_env("CLUSTER_NAME", region_var, "NAMESPACE")
cluster_name = config["CLUSTER_NAME"]
region = get_region()
namespace = config["NAMESPACE"]

# Configure kubectl
print("### Configuring kubectl\n")
configure_kubectl(cluster_name, region)
ok("kubectl configured")

# Test cluster access
result = run(["kubectl", "cluster-info"], check=True, stream=False)
print(result.stdout)


## 1. Pod Readiness Check

**Critical:** All pods must be in `Running` state with `Ready` status. This is the foundation of a healthy deployment.


In [ ]:
from shared._k8s_helpers import get_pods, wait_for_deployments_ready, require_namespace
import json

# Ensure namespace exists
require_namespace(namespace)

# Wait for deployments to be ready (with timeout)
print("### Waiting for Deployments to be Ready\n")
print("This may take a few minutes if pods are still starting...\n")

try:
    wait_for_deployments_ready(namespace, timeout="10m")
except Exception as e:
    print(f"⚠️  Timeout or error waiting for deployments: {e}")
    print("💡 Some pods may still be starting. Continuing with status check...")

# Get pod status
print("\n### Pod Status\n")
pods_output = get_pods(namespace)
print(pods_output)

# Parse pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)
pods_data = json.loads(result.stdout)

# Analyze pod status
running = 0
pending = 0
failed = 0
ready = 0
total = len(pods_data.get("items", []))

for pod in pods_data.get("items", []):
    status = pod.get("status", {})
    phase = status.get("phase", "Unknown")
    conditions = status.get("conditions", [])
    
    if phase == "Running":
        running += 1
        # Check ready condition
        for cond in conditions:
            if cond.get("type") == "Ready" and cond.get("status") == "True":
                ready += 1
                break
    elif phase == "Pending":
        pending += 1
    elif phase == "Failed":
        failed += 1

print(f"\n### Pod Summary")
print(f"Total pods: {total}")
print(f"Running: {running}")
print(f"Ready: {ready}")
print(f"Pending: {pending}")
print(f"Failed: {failed}")

if ready == total and total > 0:
    ok(f"All {total} pods are ready")
elif running == total and total > 0:
    warn(f"All pods running but {total - ready} not ready yet")
else:
    warn(f"Pod status: {running}/{total} running, {ready}/{total} ready")
    if pending > 0:
        print("💡 Some pods are still pending. Check events for issues:")
        run(["kubectl", "get", "events", "-n", namespace, "--sort-by=.lastTimestamp"], check=False, stream=True)


In [ ]:
# Check PVC status
print("### Persistent Volume Claims Status\n")

result = run(
    ["kubectl", "get", "pvc", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)
pvc_data = json.loads(result.stdout)

# Display PVCs
print("PVC Details:")
print("=" * 80)
run(["kubectl", "get", "pvc", "-n", namespace, "-o", "wide"], check=True, stream=True)
print("=" * 80)

# Analyze PVC status
bound = 0
pending = 0
total = len(pvc_data.get("items", []))

for pvc in pvc_data.get("items", []):
    status = pvc.get("status", {})
    phase = status.get("phase", "Unknown")
    
    if phase == "Bound":
        bound += 1
    elif phase == "Pending":
        pending += 1
        # Show details for pending PVCs
        name = pvc.get("metadata", {}).get("name", "unknown")
        print(f"\n⚠️  PVC '{name}' is Pending")
        print("   Common causes:")
        print("   - EBS CSI driver not installed")
        print("   - No StorageClass available")
        print("   - Insufficient storage quota")

print(f"\n### PVC Summary")
print(f"Total PVCs: {total}")
print(f"Bound: {bound}")
print(f"Pending: {pending}")

if bound == total and total > 0:
    ok(f"All {total} PVCs are bound")
elif pending > 0:
    warn(f"{pending} PVC(s) still pending - storage issue likely")
    print("💡 Check EBS CSI driver and StorageClasses")
else:
    ok("PVC status looks good")


## 3. Ingress Provisioning Check

**Critical:** The AWS ALB (Application Load Balancer) must be provisioned. This is how external traffic reaches LangSmith.

Common issue: ALB never appears due to wrong ingress assumptions.


In [ ]:
# Check ingress resources
print("### Ingress Resources\n")

# Get ingress
result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "json"],
    check=False,  # May not exist yet
    stream=False
)

if result.returncode == 0:
    ingress_data = json.loads(result.stdout)
    ingresses = ingress_data.get("items", [])
    
    if ingresses:
        print("Ingress Details:")
        print("=" * 80)
        run(["kubectl", "get", "ingress", "-n", namespace, "-o", "wide"], check=True, stream=True)
        print("=" * 80)
        
        for ingress in ingresses:
            name = ingress.get("metadata", {}).get("name", "unknown")
            status = ingress.get("status", {})
            load_balancer = status.get("loadBalancer", {})
            ingress_hosts = []
            
            # Get ingress hosts
            rules = ingress.get("spec", {}).get("rules", [])
            for rule in rules:
                host = rule.get("host", "")
                if host:
                    ingress_hosts.append(host)
            
            print(f"\nIngress: {name}")
            if ingress_hosts:
                print(f"  Hosts: {', '.join(ingress_hosts)}")
            
            # Check for ALB address
            if load_balancer.get("ingress"):
                alb_addresses = [ing.get("hostname", ing.get("ip", "")) for ing in load_balancer["ingress"]]
                if alb_addresses:
                    ok(f"ALB provisioned: {', '.join(alb_addresses)}")
                    print(f"  💡 Access LangSmith at: https://{alb_addresses[0]}")
                else:
                    warn("ALB ingress entry exists but no address found")
            else:
                warn("ALB not yet provisioned (may take a few minutes)")
                print("  💡 Wait a few minutes and check again")
    else:
        warn("No ingress resources found")
        print("💡 Ingress may not be configured in Helm values")
else:
    warn("Could not retrieve ingress resources")
    print("💡 Ingress may not exist yet or namespace is incorrect")

# Also check for ALB Ingress Controller
print("\n### ALB Ingress Controller\n")
result = run(
    ["kubectl", "get", "pods", "-n", "kube-system", "-l", "app.kubernetes.io/name=aws-load-balancer-controller", "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    controller_data = json.loads(result.stdout)
    controllers = controller_data.get("items", [])
    if controllers:
        ok(f"ALB Ingress Controller found ({len(controllers)} pod(s))")
    else:
        warn("ALB Ingress Controller not found")
        print("💡 ALB Ingress Controller must be installed for ingress to work")
else:
    warn("Could not check ALB Ingress Controller status")


## 4. Endpoint Reachability Check

Verify that services are accessible and responding. We'll check:
- Service endpoints
- Health check endpoints (if available)
- Internal service connectivity


In [ ]:
# Check services
print("### Service Endpoints\n")

result = run(
    ["kubectl", "get", "svc", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)
services_data = json.loads(result.stdout)

print("Services:")
print("=" * 80)
run(["kubectl", "get", "svc", "-n", namespace], check=True, stream=True)
print("=" * 80)

services = services_data.get("items", [])
if services:
    ok(f"Found {len(services)} service(s)")
    
    # Check for LoadBalancer services
    lb_services = [svc for svc in services if svc.get("spec", {}).get("type") == "LoadBalancer"]
    if lb_services:
        print(f"\nLoadBalancer services: {len(lb_services)}")
        for svc in lb_services:
            name = svc.get("metadata", {}).get("name", "unknown")
            status = svc.get("status", {}).get("loadBalancer", {})
            if status.get("ingress"):
                lb_address = status["ingress"][0].get("hostname") or status["ingress"][0].get("ip")
                ok(f"Service '{name}' has LoadBalancer: {lb_address}")
            else:
                warn(f"Service '{name}' LoadBalancer pending")
    
    # Test internal connectivity (if we can exec into a pod)
    print("\n### Testing Internal Service Connectivity\n")
    # Try to find a pod we can exec into
    result = run(
        ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[0].metadata.name}"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0 and result.stdout.strip():
        test_pod = result.stdout.strip()
        print(f"Testing connectivity from pod: {test_pod}")
        # Try a simple DNS lookup or curl
        # This is a basic check - actual health endpoints depend on the application
        print("💡 Internal connectivity tests depend on application-specific health endpoints")
else:
    warn("No services found")


## 5. Basic UI Availability Check

**Final validation:** Can we actually access the LangSmith UI through the ingress?

This is the ultimate test - if the UI loads, everything is working.


In [ ]:
import requests
from urllib.parse import urlparse

# Get ingress hostname
print("### UI Availability Check\n")

result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "jsonpath={.items[0].status.loadBalancer.ingress[0].hostname}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    ingress_host = result.stdout.strip()
    print(f"Ingress hostname: {ingress_host}")
    
    # Try to access the UI (HTTPS)
    ui_url = f"https://{ingress_host}"
    print(f"\nTesting UI availability at: {ui_url}")
    print("(This may take a moment if ALB is still provisioning...)\n")
    
    try:
        # Use a short timeout and allow redirects
        response = requests.get(ui_url, timeout=10, allow_redirects=True, verify=False)
        if response.status_code == 200:
            ok(f"UI is accessible! Status: {response.status_code}")
            print(f"💡 Open in browser: {ui_url}")
        elif response.status_code in [301, 302, 307, 308]:
            ok(f"UI redirects (status: {response.status_code}) - likely working")
            print(f"💡 Redirect location: {response.headers.get('Location', 'N/A')}")
            print(f"💡 Open in browser: {ui_url}")
        else:
            warn(f"UI returned status {response.status_code}")
            print("💡 UI may still be starting or there may be a configuration issue")
    except requests.exceptions.SSLError:
        # SSL errors might be expected if using self-signed certs
        warn("SSL verification failed (may be expected with self-signed certs)")
        print(f"💡 Try accessing: {ui_url}")
        print("   Browser may show security warning - this is normal for self-signed certs")
    except requests.exceptions.Timeout:
        warn("UI request timed out")
        print("💡 ALB may still be provisioning, or ingress is not fully configured")
        print(f"   Try again in a few minutes: {ui_url}")
    except requests.exceptions.ConnectionError as e:
        warn(f"Could not connect to UI: {e}")
        print("💡 ALB may still be provisioning")
        print(f"   Check AWS console for ALB status, then try: {ui_url}")
    except Exception as e:
        warn(f"Error accessing UI: {e}")
        print(f"💡 Manual check: Open {ui_url} in a browser")
else:
    warn("Could not determine ingress hostname")
    print("💡 Ingress may not be provisioned yet")
    print("   Run the ingress check above and wait for ALB to be created")


## Collecting Diagnostic Artifacts

Save cluster state snapshots for future troubleshooting reference.


In [ ]:
from datetime import datetime

# Create diagnostic snapshot
print("### Collecting Diagnostic Artifacts\n")

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
diagnostics_dir = artifacts_dir / f"diagnostics-{timestamp}"
diagnostics_dir.mkdir(exist_ok=True)

print(f"Saving diagnostics to: {diagnostics_dir}\n")

# Save various cluster states
diagnostics = [
    ("pods", ["kubectl", "get", "pods", "-n", namespace, "-o", "yaml"]),
    ("services", ["kubectl", "get", "svc", "-n", namespace, "-o", "yaml"]),
    ("ingress", ["kubectl", "get", "ingress", "-n", namespace, "-o", "yaml"]),
    ("pvc", ["kubectl", "get", "pvc", "-n", namespace, "-o", "yaml"]),
    ("deployments", ["kubectl", "get", "deployments", "-n", namespace, "-o", "yaml"]),
    ("events", ["kubectl", "get", "events", "-n", namespace, "--sort-by=.lastTimestamp"]),
]

for name, cmd in diagnostics:
    try:
        result = run(cmd, check=False, stream=False)
        output_file = diagnostics_dir / f"{name}.txt"
        with open(output_file, "w") as f:
            f.write(result.stdout)
            if result.stderr:
                f.write("\n\nSTDERR:\n")
                f.write(result.stderr)
        print(f"✅ Saved {name}")
    except Exception as e:
        print(f"⚠️  Could not save {name}: {e}")

ok(f"Diagnostics saved to: {diagnostics_dir}")
print("\n💡 These artifacts can be used for troubleshooting or support tickets")


## Go/No-Go Checklist

Review this checklist. All items should be ✅ before considering the deployment ready.

### ✅ Validation Checklist

- [ ] All pods are running and ready
- [ ] All PVCs are bound
- [ ] Ingress/ALB is provisioned
- [ ] Services are accessible
- [ ] UI is reachable (or ALB is provisioning)
- [ ] Diagnostic artifacts collected

### 🎯 Next Steps

**If all checks pass:**
- ✅ You have a working baseline deployment
- ✅ You're on a supported path
- ✅ Ready to proceed to Module 2 (SSO/OIDC configuration)

**If checks fail:**
- Review the warnings above
- Check diagnostic artifacts
- Common issues:
  - **PVCs pending:** EBS CSI driver not installed
  - **ALB not appearing:** Wrong ingress configuration
  - **Pods not ready:** Check events and logs
  - **UI not accessible:** Wait for ALB provisioning (can take 5-10 minutes)

### 📋 Baseline Reference

This validation checklist becomes your **baseline reference** for future troubleshooting. Save the diagnostic artifacts and refer back to this state when investigating issues.
